# Logistic Regression — Smart MCQ Solver


## Imports


In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
import warnings
warnings.filterwarnings('ignore')


## Wandb Setup


In [2]:
!pip install -q wandb


In [3]:
import wandb

WANDB_API_KEY = "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"
WANDB_ENTITY = "models-indian-institute-of-technology-madras481" 
WANDB_PROJECT = "23f3001092-t22026"    

wandb.login(key=WANDB_API_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

## Load Data


In [4]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()


Train shape: (2000, 8)
Test shape: (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## Evaluation Metrics


In [5]:
def apk(actual, predicted, k=3):
    score, hits = 0.0, 0
    for i, p in enumerate(predicted[:k]):
        if p == actual:
            hits += 1
            score += hits / (i + 1.0)
    return score

def mapk(actuals, preds):
    return float(np.mean([apk(a, p) for a, p in zip(actuals, preds)]))


## Pairwise Data Preparation


In [6]:
rows = []
for _, r in train.iterrows():
    for c in OPTION_COLS:
        rows.append({
            'text':   str(r['prompt']) + ' ' + str(r[c]),
            'qid':    r['id'],
            'option': c,
            'label':  1 if r['answer'] == c else 0
        })
pw_train = pd.DataFrame(rows)
print("Train pairwise shape:", pw_train.shape)
pw_train.head()


Train pairwise shape: (10000, 4)


,text,qid,option,label
0,Pick the best possible answer: What is Martin ...,1,A,0
1,Pick the best possible answer: What is Martin ...,1,B,1
2,Pick the best possible answer: What is Martin ...,1,C,0
3,Pick the best possible answer: What is Martin ...,1,D,0
4,Pick the best possible answer: What is Martin ...,1,E,0


In [7]:
rows = []
for _, r in test.iterrows():
    for c in OPTION_COLS:
        rows.append({
            'text':   str(r['prompt']) + ' ' + str(r[c]),
            'qid':    r['id'],
            'option': c
        })
pw_test = pd.DataFrame(rows)
print("Test pairwise shape:", pw_test.shape)
pw_test.head()


Test pairwise shape: (2500, 3)


,text,qid,option
0,Pick the best possible answer: What is the rel...,1,A
1,Pick the best possible answer: What is the rel...,1,B
2,Pick the best possible answer: What is the rel...,1,C
3,Pick the best possible answer: What is the rel...,1,D
4,Pick the best possible answer: What is the rel...,1,E


## TF-IDF Features


In [8]:
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
X_tr  = tfidf.fit_transform(pw_train['text'])
X_tst = tfidf.transform(pw_test['text'])
y_tr  = pw_train['label'].values
groups = pw_train['qid'].values

print("Train features:", X_tr.shape)
print("Test features:", X_tst.shape)


Train features: (10000, 13679)
Test features: (2500, 13679)


## Helper Functions


In [9]:
def rank_preds(pw, scores, orig_df):
    pw = pw.copy()
    pw['score'] = scores
    return [
        pw[pw['qid'] == qid].sort_values('score', ascending=False)['option'].tolist()[:3]
        for qid in orig_df['id']
    ]


## Training with Wandb Logging


In [10]:
gkf    = GroupKFold(n_splits=5)
oof    = np.zeros(len(pw_train))
tst_lr = np.zeros(len(pw_test))

run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name="logistic-regression-tfidf",
    config={
        "model": "LogisticRegression",
        "C": 4.0,
        "max_iter": 1000,
        "tfidf_max_features": 50000,
        "ngram_range": "(1,2)",
        "n_splits": 5,
    },
)


wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260714_185901-pffhqiza
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run logistic-regression-tfidf
wandb: ⭐️ View project at https://wandb.ai/models-indian-institute-of-technology-madras481/23f3001092-t22026
wandb: 🚀 View run at https://wandb.ai/models-indian-institute-of-technology-madras481/23f3001092-t22026/runs/pffhqiza


In [11]:
for fold, (ti, vi) in enumerate(gkf.split(X_tr, y_tr, groups)):
    lr = LogisticRegression(C=4.0, max_iter=1000, n_jobs=-1)
    lr.fit(X_tr[ti], y_tr[ti])
    oof[vi] = lr.predict_proba(X_tr[vi])[:, 1]
    tst_lr += lr.predict_proba(X_tst)[:, 1] / 5

    val_qids   = pw_train.iloc[vi]['qid'].unique()
    val_train  = train[train['id'].isin(val_qids)]
    val_pw     = pw_train[pw_train['qid'].isin(val_qids)]
    val_scores = oof[pw_train['qid'].isin(val_qids).values]
    fold_map3  = mapk(val_train['answer'].tolist(), rank_preds(val_pw, val_scores, val_train))
    print(f"Fold {fold+1}  MAP@3={fold_map3:.4f}")

    wandb.log({"fold": fold + 1, "fold_map3": fold_map3})


Fold 1  MAP@3=0.9925
Fold 2  MAP@3=0.9800
Fold 3  MAP@3=0.9850
Fold 4  MAP@3=0.9900
Fold 5  MAP@3=0.9875


In [12]:
oof_map3 = mapk(train['answer'].tolist(), rank_preds(pw_train, oof, train))
print(f"OOF MAP@3: {oof_map3:.4f}")

wandb.log({"oof_map3": oof_map3})
wandb.finish()


wandb: uploading summary; updating run metadata


OOF MAP@3: 0.9870


wandb: uploading summary
wandb: uploading summary; uploading wandb-summary.json; uploading config.yaml
wandb: uploading summary
wandb: uploading history steps 0-5, summary, console lines 0-5
wandb: 
wandb: Run history:
wandb:      fold ▁▃▅▆█
wandb: fold_map3 █▁▄▇▅
wandb:  oof_map3 ▁
wandb: 
wandb: Run summary:
wandb:      fold 5
wandb: fold_map3 0.9875
wandb:  oof_map3 0.987
wandb: 
wandb: 🚀 View run logistic-regression-tfidf at: https://wandb.ai/models-indian-institute-of-technology-madras481/23f3001092-t22026/runs/pffhqiza
wandb: ⭐️ View project at: https://wandb.ai/models-indian-institute-of-technology-madras481/23f3001092-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260714_185901-pffhqiza/logs


## Submission


In [13]:
test_preds = rank_preds(pw_test, tst_lr, test)

submission = pd.DataFrame({
    'ID':         test['id'],
    'Prediction': [' '.join(p) for p in test_preds]
})

submission.to_csv('submission.csv', index=False)
print(submission.head(10).to_string(index=False))
print("submission.csv saved")


 ID Prediction
  1      A E B
  2      B E A
  3      B E D
  4      E C D
  5      C A B
  6      D A B
  7      E D C
  8      B E A
  9      C D E
 10      B A C
submission.csv saved
